In [ ]:
# 1. Cấu hình bảng giá trị mốc BP cho từng kim loại (trích xuất từ Bảng 2 trong ảnh)
# Thứ tự các giá trị tương ứng với i = 1, 2, 3, 4, 5
metal_configs = {
    'As_mg_L':   [0.01, 0.02, 0.05, 0.1,  0.1],      # Lưu ý: As mức 5 là >0.1
    'Cd_mg_L':   [0.005, 0.005, 0.008, 0.01, 0.1],
    'Pb_mg_L':   [0.02, 0.02, 0.04, 0.05, 0.5],
    'CrVI': [0.01, 0.02, 0.04, 0.05, 0.1],
    'Cu':   [0.1, 0.2, 0.5, 1.0, 2.0],
    'Zn':   [0.5, 1.0, 1.5, 2.0, 3.0],
    'Hg_mg_L':   [0.001, 0.001, 0.0015, 0.002, 0.01]
}

# Giá trị q tương ứng với các mức i từ 1 đến 5
q_values = [100, 75, 50, 25, 10]

# 2. Hàm tính WQI tổng quát
def calculate_wqi_generic(cp, bp_list):
    if pd.isna(cp) or not isinstance(cp, (int, float)):
        return np.nan

    # Nếu Cp tốt hơn hoặc bằng mức chuẩn cao nhất (mức 1)
    if cp <= bp_list[0]:
        return q_values[0]

    # Nếu Cp tệ hơn hoặc bằng mức thấp nhất (mức 5)
    if cp >= bp_list[4]:
        return q_values[4]

    # Tìm khoảng để áp dụng Công thức 1
    for i in range(len(bp_list) - 1):
        bp_i, bp_i1 = bp_list[i], bp_list[i+1]
        q_i, q_i1 = q_values[i], q_values[i+1]

        if bp_i <= cp <= bp_i1:
            # Trường hợp bp_i == bp_i1 (như Pb, Hg ở mức 1-2)
            if cp == bp_i and i != 0:
                return cp
            if cp == bp_i1 and i != 4:
                return cp

            # Áp dụng Công thức 1
            wqi = ((q_i - q_i1) / (bp_i1 - bp_i)) * (bp_i1 - cp) + q_i1
            return round(wqi, 2)

# 3. Vòng lặp tự động tính cho tất cả các cột có trong cấu hình
for metal, bp_points in metal_configs.items():
    if metal in wqi_data_convert_cell.columns:
        new_col_name = f'WQI_{metal.split()[0]}' # Tạo tên cột như WQI_As, WQI_Cd...
        wqi_data_convert_cell[new_col_name] = wqi_data_convert_cell[metal].apply(
            lambda x: calculate_wqi_generic(x, bp_points)
        )

# Hiển thị kết quả kiểm tra 5 dòng đầu
print(wqi_data_convert_cell.head())

In [ ]:
def calculate_wqi_do(do_measured, temp):
    """
    Tính WQI cho thông số DO dựa trên nhiệt độ và nồng độ DO hòa tan.
    do_measured: Cột 'DO mg/l'
    temp: Cột 'Nhiệt độ (0C)'
    """
    if pd.isna(do_measured) or pd.isna(temp) or not isinstance(do_measured, (int, float)) or not isinstance(temp, (int, float)):
        return np.nan

    # Bước 1: Tính DO bão hòa (DO_baohua) dựa trên nhiệt độ T
    # Công thức: 14.652 - 0.41022*T + 0.0079910*T^2 - 0.0000777747*T^3
    do_sat = (14.652 - 0.41022 * temp +
              0.0079910 * (temp**2) -
              0.000077774 * (temp**3))

    # Tính giá trị DO % bão hòa (Cp)
    cp = (do_measured / do_sat) * 100

    # Bảng 3: Quy định giá trị BPi và qi đối với DO % bão hòa
    bp_do = [20, 50, 75, 88, 112, 125, 150, 200]
    q_do = [25, 50, 75, 100, 100, 75, 50, 25]

    # Kiểm tra nếu Cp trùng đúng với các giá trị mốc trong bảng
    for i, bp_val in enumerate(bp_do):
        if round(cp, 2) == bp_val:
            return q_do[i]

    # Các điều kiện đặc biệt theo hướng dẫn
    if cp < 20 or cp > 200:
        return 10

    if 88 <= cp <= 112:
        return 100

    # Trường hợp 20 < Cp < 88: Sử dụng Công thức 2
    if 20 < cp < 88:
        # Xác định khoảng i
        for i in range(0, 3): # Chỉ xét các mốc 20, 50, 75, 88
            bp_i, bp_i1 = bp_do[i], bp_do[i+1]
            q_i, q_i1 = q_do[i], q_do[i+1]

            if bp_i < cp < bp_i1:
                # Công thức 2: WQI = [(qi+1 - qi) / (BPi+1 - BPi)] * (Cp - BPi) + qi
                wqi = ((q_i1 - q_i) / (bp_i1 - bp_i)) * (cp - bp_i) + q_i
                return round(wqi, 2)

    # Trường hợp 112 < Cp < 200: Sử dụng Công thức 1
    if 112 < cp < 200:
        # Xác định khoảng i (các mốc 112, 125, 150, 200)
        for i in range(4, 7):
            bp_i, bp_i1 = bp_do[i], bp_do[i+1]
            q_i, q_i1 = q_do[i], q_do[i+1]

            if bp_i < cp < bp_i1:
                # Công thức 1: WQI = [(qi - qi+1) / (BPi+1 - BPi)] * (BPi+1 - Cp) + qi+1
                wqi = ((q_i - q_i1) / (bp_i1 - bp_i)) * (bp_i1 - cp) + q_i1
                return round(wqi, 2)


# Áp dụng cho DataFrame
# Lưu ý: Thay đổi tên cột 'Nhiệt độ (0C)' nếu trong file của bạn đặt tên khác
wqi_data_convert_cell['WQI_DO'] = wqi_data_convert_cell.apply(
    lambda row: calculate_wqi_do(row['DO_mg_L'], row['Temperature_C']), axis=1
)

print(wqi_data_convert_cell[['DO_mg_L', 'Temperature_C', 'WQI_DO']].head())

In [ ]:
def calculate_wqi_ph(cp):
    """
    Tính WQI cho thông số pH dựa trên Bảng 4 và các công thức nội suy.
    cp: giá trị pH quan trắc được.
    """
    if pd.isna(cp) or not isinstance(cp, (int, float)):
        return np.nan

    # Bảng 4: Quy định giá trị BPi và qi đối với pH
    # i=1: <5.5 (q=10), i=2: 5.5 (q=50), i=3: 6 (q=100)
    # i=4: 8.5 (q=100), i=5: 9 (q=50), i=6: >9 (q=10)

    # 1. Trường hợp nằm ngoài khoảng hoặc trùng mốc biên cực đoan
    if cp < 5.5 or cp > 9:
        return 10


    # 2. Trường hợp trùng khớp với các giá trị mốc BP trong bảng
    if cp == 5.5: return 50
    if cp == 6.0: return 100
    if cp == 8.5: return 100
    if cp == 9.0: return 50

    # 3. Trường hợp 5.5 < pH < 6: Sử dụng Công thức 2
    if 5.5 < cp < 6:
        # i=2 (BPi=5.5, qi=50); i+1=3 (BPi+1=6, qi+1=100)
        bp_i, bp_i1 = 5.5, 6.0
        q_i, q_i1 = 50, 100
        # WQI = [(qi+1 - qi) / (BPi+1 - BPi)] * (Cp - BPi) + qi
        wqi = ((q_i1 - q_i) / (bp_i1 - bp_i)) * (cp - bp_i) + q_i
        return round(wqi, 2)

    # 4. Trường hợp 6 <= pH <= 8.5: WQI = 100
    if 6 <= cp <= 8.5:
        return 100

    # 5. Trường hợp 8.5 < pH < 9: Sử dụng Công thức 1
    if 8.5 < cp < 9:
        # i=4 (BPi=8.5, qi=100); i+1=5 (BPi+1=9, qi+1=50)
        bp_i, bp_i1 = 8.5, 9.0
        q_i, q_i1 = 100, 50
        # WQI = [(qi - qi+1) / (BPi+1 - BPi)] * (BPi+1 - Cp) + qi+1
        wqi = ((q_i - q_i1) / (bp_i1 - bp_i)) * (bp_i1 - cp) + q_i1
        return round(wqi, 2)

# Áp dụng cho toàn bộ dữ liệu trong cột 'pH'
wqi_data_convert_cell['WQI_ph'] = wqi_data_convert_cell['pH'].apply(calculate_wqi_ph)

# Hiển thị kết quả kiểm tra
print(wqi_data_convert_cell[['pH', 'WQI_ph']].head())

In [ ]:
# 1. Cấu hình bảng giá trị mốc BP cho từng thông số (theo Bảng 1 trong ảnh)
# Thứ tự q tương ứng: 100, 75, 50, 25, 10
q_values = [100, 75, 50, 25, 10]

params_config = {
    'BOD5_mg_L':          [4, 6, 15, 25, 50],
    'COD_mg_L':           [10, 15, 30, 50, 150],
    #'TOC':                [4, 6, 15, 25, 50],
    'NH4_mg_L':          [0.3, 0.3, 0.6, 0.9, 5],   # BP1=0.3 (L1 là <0.3)
    'NO3_mg_L':           [2, 5, 10, 15, 15],        # L5 là >15
    'PO4_mg_L':         [0.1, 0.2, 0.3, 0.5, 4],
    'Coliform_MPN_100ml': [2500, 5000, 7500, 10000, 10000], # L5 là >10000
    'EColi':             [20, 50, 100, 200, 200]    # L5 là >200
}

# 2. Hàm tính WQI SI tổng quát (Áp dụng Công thức 1)
def calculate_wqi_formula1(cp, bp_list):
    if pd.isna(cp): return np.nan

    # Trường hợp trùng với mốc biên hoặc vượt ngưỡng cực đoan
    if cp <= bp_list[0]: return q_values[0]
    if cp >= bp_list[4]: return q_values[4]

    # Kiểm tra nếu giá trị Cp trùng đúng với điểm mốc BPi
    for i, bp_val in enumerate(bp_list):
        if cp == bp_val:
            return q_values[i]

    # Áp dụng Công thức 1 cho các khoảng giữa
    for i in range(len(bp_list) - 1):
        bp_i, bp_i1 = bp_list[i], bp_list[i+1]
        q_i, q_i1 = q_values[i], q_values[i+1]

        if bp_i < cp < bp_i1:
            # WQI = [(qi - qi+1) / (BPi+1 - BPi)] * (BPi+1 - Cp) + qi+1
            wqi = ((q_i - q_i1) / (bp_i1 - bp_i)) * (bp_i1 - cp) + q_i1
            return round(wqi, 2)

# 3. Vòng lặp tự động tính cho các cột (Trừ NO2- xử lý riêng)
for param, bp_points in params_config.items():
    if param in wqi_data_convert_cell.columns:
        # Tạo tên cột mới, ví dụ: WQI_BOD5, WQI_NH4...
        new_col = f"WQI_{param.replace('+', '').replace('-', '')}"
        wqi_data_convert_cell[new_col] = wqi_data_convert_cell[param].apply(
            lambda x: calculate_wqi_formula1(x, bp_points)
        )

# 4. Xử lý riêng cho NO2- (Vì bảng chỉ có 2 mức: <=0.05 và >0.05)
if 'NO2_mg_L' in wqi_data_convert_cell.columns:
    wqi_data_convert_cell['WQI_NO2_mg_L'] = wqi_data_convert_cell['NO2_mg_L'].apply(
        lambda x: 100 if x <= 0.05 else 10 if x > 0.05 else np.nan
    )

# Hiển thị kết quả các cột WQI vừa tạo
wqi_cols = [c for c in wqi_data_convert_cell.columns if c.startswith('WQI_')]
print(wqi_data_convert_cell[wqi_cols].head())

In [ ]:
def calculate_final_vn_wqi(row):
    # 1. Phân nhóm các thông số (Dựa trên image_8a4caa.png)
    group_i_cols = ['WQI_ph']

    # Nhóm II: Thuốc bảo vệ thực vật (Nếu bạn có dữ liệu này)
    group_ii_cols = [col for col in row.index if col.startswith('WQI_') and
                     any(p in col for p in ['Aldrin', 'DDTs', 'BHC', 'Dieldrin'])]

    # Nhóm III: Kim loại nặng
    group_iii_cols = ['WQI_As_mg_L', 'WQI_Cd_mg_L', 'WQI_Pb_mg_L', 'WQI_CrVI', 'WQI_Cu', 'WQI_Zn', 'WQI_Hg_mg_L']

    # Nhóm IV: Hữu cơ & Dinh dưỡng (Bắt buộc - ít nhất 3 thông số)
    group_iv_cols = ['WQI_DO', 'WQI_BOD5_mg_L', 'WQI_COD_mg_L', 'WQI_TOC', 'NH4_mg_L', 'NO3_mg_L', 'WQI_NO2_mg_L', 'WQI_PO4_mg_L']

    # Nhóm V: Vi sinh
    group_v_cols = ['WQI_Coliform_MPN_100ml', 'WQI_EColi']

    # 2. Kiểm tra điều kiện tính toán
    # - Bắt buộc có nhóm IV và nhóm IV phải có ít nhất 3 thông số
    valid_iv = [row[c] for c in group_iv_cols if c in row.index and pd.notna(row[c])]
    if len(valid_iv) < 3:
        return np.nan # Không đủ điều kiện tính VN_WQI

    # - Phải có tối thiểu 3/5 nhóm thông số
    available_groups = 0
    groups_data = {}

    for g_name, cols in [('I', group_i_cols), ('II', group_ii_cols),
                         ('III', group_iii_cols), ('IV', group_iv_cols), ('V', group_v_cols)]:
        valid_vals = [row[c] for c in cols if c in row.index and pd.notna(row[c])]
        if valid_vals:
            available_groups += 1
            groups_data[g_name] = valid_vals

    if available_groups < 3:
        return np.nan

    # 3. Tính toán giá trị trung bình cho từng nhóm
    # Nhóm I: WQI_I
    wqi_1 = groups_data['I'][0] if 'I' in groups_data else 100

    # Nhóm II & III: Trung bình nhân (Geometric Mean)
    wqi_2_val = gmean(groups_data['II']) if 'II' in groups_data else 100
    wqi_3_val = gmean(groups_data['III']) if 'III' in groups_data else 100

    # Nhóm IV & V: Trung bình cộng (Arithmetic Mean)
    wqi_4_avg = np.mean(groups_data['IV'])
    wqi_5_avg = np.mean(groups_data['V']) if 'V' in groups_data else np.nan

    # 4. Áp dụng Công thức 3
    # Thành phần căn bậc hai của nhóm IV và V
    if not np.isnan(wqi_5_avg):
        bracket_term = np.sqrt(wqi_4_avg * wqi_5_avg)
    else:
        bracket_term = wqi_4_avg # Nếu thiếu nhóm V, chỉ dùng nhóm IV

    final_wqi = (wqi_1 / 100) * (wqi_2_val / 100) * (wqi_3_val / 100) * bracket_term

    return round(final_wqi, 0)

# Áp dụng tính toán cho toàn bộ DataFrame
wqi_data_convert_cell['VN_WQI'] = wqi_data_convert_cell.apply(calculate_final_vn_wqi, axis=1)

# Hiển thị kết quả cuối cùng
print(wqi_data_convert_cell[['VN_WQI']].head())